# Tech Addiction Prediction: CatBoost Model
In this notebook, we build a robust model using `CatBoost`, which is uniquely powerful for datasets with categorical features.
We include our V2 Feature Engineering, run Optuna for hyperparameter tuning, and finish with a 5-Fold Stratified Cross Validation ensemble.


In [ ]:
import pandas as pd
import numpy as np
import catboost as cb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')


## 1. Data Loading
We load the data from the standard Kaggle input directory.


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values

print(f"Train features shape: {X.shape}")
print(f"Test features shape: {test_df.drop(['id'], axis=1).shape}")


## 2. Feature Engineering & Preprocessing
We apply our V2 features. Unlike XGBoost and LightGBM, we do NOT one-hot encode our categorical variables. CatBoost handles them natively and mathematically optimally when passed as strings.


In [ ]:
def engineer_features(train, test):
    train = train.copy()
    test = test.copy()
    
    for df in [train, test]:
        # Original V2 Features
        df['weekend_delta'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
        df['social_media_prop'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['unaccounted_screen_time'] = df['daily_screen_time_hours'] - (df['social_media_hours'] + df['gaming_hours'] + df['work_study_hours'])
        
        # NEW V5 Features
        # Ratios
        df['productivity_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['entertainment_ratio'] = (df['social_media_hours'] + df['gaming_hours']) / (df['daily_screen_time_hours'] + 1e-5)
        df['screen_to_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-5)
        
        # Engagement Intensity
        df['interaction_intensity'] = df['notifications_per_day'] * df['app_opens_per_day']
        
        # Flags & Binning
        df['sleep_deprived'] = (df['sleep_hours'] < 6.5).astype(int)
        df['age_group'] = pd.cut(df['age'], bins=[0, 20, 30, 40, 50, 100], labels=False)
        
    return train, test

print("Engineering Deep V5 features...")
X, X_test = engineer_features(X, test_df.drop(['id'], axis=1))
print(f"New Train features shape: {X.shape}")

categorical_features = ['gender', 'academic_work_impact', 'stress_level', 'age_group']
for col in categorical_features:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)


## 3. Optuna Hyperparameter Tuning
We search for the best CatBoost hyperparameters using a fast 3-Fold CV.

In [ ]:
def objective(trial):
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'iterations': trial.suggest_int('iterations', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bootstrap_type': 'Bernoulli',
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'verbose': False,
        'task_type': 'GPU'
    }
    
    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_auc = []
    
    for train_idx, val_idx in skf_tune.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]
        
        model = cb.CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=categorical_features,
            early_stopping_rounds=50,
            verbose=False
        )
        preds = model.predict_proba(X_va)[:, 1]
        oof_auc.append(roc_auc_score(y_va, preds))
        
    return np.mean(oof_auc)

print("Starting Optuna tuning...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=150)
print("Best params found:", study.best_params)


## 4. Stratified 5-Fold Cross Validation
We evaluate the model using 5 folds to generate Out-Of-Fold (OOF) predictions and use early stopping during training.


In [ ]:
print("Starting 5-Fold Stratified Cross-Validation for CatBoost...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models = []

cb_params = study.best_params.copy()
cb_params['loss_function'] = 'Logloss'
cb_params['eval_metric'] = 'AUC'
cb_params['random_seed'] = 42
cb_params['verbose'] = False
cb_params['task_type'] = 'GPU'

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = cb.CatBoostClassifier(**cb_params)
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=categorical_features,
        early_stopping_rounds=100,
        verbose=False
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    models.append(model)

print("\nCross-Validation complete!")


### Metric Evaluation


In [ ]:
print("Evaluating OOF predictions...")
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("CatBoost Model (5-Fold OOF) Performance:")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)


## 5. Inference and Submission


In [ ]:
print("Predicting on test set using Fold models...")
test_preds_proba = np.zeros(len(X_test))

for model in models:
    test_preds_proba += model.predict_proba(X_test)[:, 1] / len(models)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
display(submission.head())
